In [1]:
# Read the annotations file
import pandas as pd
df_all = pd.read_csv('../data/raw-data/slice_annotation.csv', index_col = 0)

print(df_all.shape)
(df_all.head())

(390, 5)


,patient_id,C3_Manual_slice,Datatype,Z_spacing,XY_spacing
0,mdacc_HNSCC-01-0216_CT-SIM-08-10-2008-_raw_raw...,192,val,NaN,NaN
0,mdacc_HNSCC-01-0217_CT-SIM-12-16-2012-_raw_raw...,131,train,NaN,NaN
0,mdacc_HNSCC-01-0218_CT-SIM-09-13-2009-_raw_raw...,166,train,NaN,NaN
0,mdacc_HNSCC-01-0219_CT-SIM-09-13-2009-_raw_raw...,150,train,NaN,NaN
0,mdacc_HNSCC-01-0220_CT-SIM-11-13-2011-_raw_raw...,129,train,NaN,NaN


In [2]:
# Apply window utility - used mainly in selection training data prep
def apply_window(image, win_centre= 40, win_width= 400):
    range_bottom = win_centre - win_width / 2
    scale = 256 / win_width
    image = image - range_bottom

    image = image * scale
    image[image < 0] = 0
    image[image > 255] = 255

    return image


In [4]:
#### Slice prediction model data preparation - TRAINING
# The z-offset for a slice should represent its offset above or below the C3 slice in mm 
# Slices below the chosen C3 slice should be given negative offsets.
import matplotlib.pyplot as plt
import functools
from skimage.transform import resize
import SimpleITK as sitk
import numpy as np

df_to_be_writen = pd.DataFrame()
df_train = df_all[df_all.Datatype.isin(['train'])]
image_dir = '../data/raw-data/scans/'
targ_dir = '../data/train/train_prediction/predcition_npy/train/'
csv_write_path = '../data/train/train_selection/prediction_meta/train.csv'
j = 0
for idx in range(df_train.shape[0]):
    image_path = image_dir + df_train.iloc[idx,0]+'.nrrd'
    patient_id =df_train.iloc[idx,0][:37]
    c3_slice = df_train.iloc[idx,1]
    print(patient_id,image_path,c3_slice,targ_dir)
     
    image_sitk =  sitk.ReadImage(image_path)
    image_array  = sitk.GetArrayFromImage(image_sitk)
    spacing =  image_sitk.GetSpacing()[2]
    
    image_array = apply_window(image_array)
    windowed_images = image_array

    print(idx+1,'th',patient_id ,image_array.shape,np.max(image_array),spacing)

    resize_func = functools.partial(resize, output_shape=[256,256],
                                    preserve_range=True, anti_aliasing=True, mode='constant')
    series = np.dstack([resize_func(im) for im in windowed_images])
    series = np.transpose(series[:, :, :, np.newaxis], [2, 0, 1, 3])
#     print(type(series),series.shape)
    for slice_idx in range(image_array.shape[0]):
        offset = spacing*(slice_idx-c3_slice)
        offset = round(offset,5)
        npy_name = str(j).zfill(6)+'_'+patient_id+'.npy'
        npy_path = targ_dir +npy_name
        im_array = series[slice_idx,:,:,:].astype(np.uint8)
        np.save(npy_path,im_array)
#         print(npy_path ,im_array.shape,'spacing',spacing)

        df_offset = pd.DataFrame({'NPY_name':npy_name,
                       'ZOffset':offset},index=[0])
        #df_to_be_writen = df_to_be_writen.append(df_offset)
        df_to_be_writen = pd.concat([df_to_be_writen,df_offset])
        df_to_be_writen.to_csv(csv_write_path)   
        j=j+1
    print()


mdacc_HNSCC-01-0217_CT-SIM-12-16-2012 ../data/raw-data/scans/mdacc_HNSCC-01-0217_CT-SIM-12-16-2012-_raw_raw_raw_xx.nrrd 131 ../data/train/train_selection/selection_npy/train/
1 th mdacc_HNSCC-01-0217_CT-SIM-12-16-2012 (260, 512, 512) 255.0 1.0

mdacc_HNSCC-01-0218_CT-SIM-09-13-2009 ../data/raw-data/scans/mdacc_HNSCC-01-0218_CT-SIM-09-13-2009-_raw_raw_raw_xx.nrrd 166 ../data/train/train_selection/selection_npy/train/
2 th mdacc_HNSCC-01-0218_CT-SIM-09-13-2009 (294, 512, 512) 255.0 1.0

mdacc_HNSCC-01-0219_CT-SIM-09-13-2009 ../data/raw-data/scans/mdacc_HNSCC-01-0219_CT-SIM-09-13-2009-_raw_raw_raw_xx.nrrd 150 ../data/train/train_selection/selection_npy/train/
3 th mdacc_HNSCC-01-0219_CT-SIM-09-13-2009 (240, 512, 512) 255.0 1.0

mdacc_HNSCC-01-0220_CT-SIM-11-13-2011 ../data/raw-data/scans/mdacc_HNSCC-01-0220_CT-SIM-11-13-2011-_raw_raw_raw_xx.nrrd 129 ../data/train/train_selection/selection_npy/train/
4 th mdacc_HNSCC-01-0220_CT-SIM-11-13-2011 (343, 512, 512) 255.0 1.0

mdacc_HNSCC-01-0222_

34 th mdacc_HNSCC-01-0263_CT-SIM-06-11-2006 (251, 512, 512) 255.0 1.0

mdacc_HNSCC-01-0264_CT-SIM-06-11-2006 ../data/raw-data/scans/mdacc_HNSCC-01-0264_CT-SIM-06-11-2006-_raw_raw_raw_xx.nrrd 142 ../data/train/train_selection/selection_npy/train/
35 th mdacc_HNSCC-01-0264_CT-SIM-06-11-2006 (254, 512, 512) 255.0 1.0

mdacc_HNSCC-01-0265_CT-SIM-06-11-2006 ../data/raw-data/scans/mdacc_HNSCC-01-0265_CT-SIM-06-11-2006-_raw_raw_raw_xx.nrrd 153 ../data/train/train_selection/selection_npy/train/
36 th mdacc_HNSCC-01-0265_CT-SIM-06-11-2006 (250, 512, 512) 255.0 1.0

mdacc_HNSCC-01-0267_CT-SIM-06-11-2006 ../data/raw-data/scans/mdacc_HNSCC-01-0267_CT-SIM-06-11-2006-_raw_raw_raw_xx.nrrd 145 ../data/train/train_selection/selection_npy/train/
37 th mdacc_HNSCC-01-0267_CT-SIM-06-11-2006 (270, 512, 512) 255.0 1.0

mdacc_HNSCC-01-0268_CT-SIM-06-11-2006 ../data/raw-data/scans/mdacc_HNSCC-01-0268_CT-SIM-06-11-2006-_raw_raw_raw_xx.nrrd 38 ../data/train/train_selection/selection_npy/train/
38 th mdacc_HNSCC

68 th mdacc_HNSCC-01-0310_CT-SIM-07-08-2007 (292, 512, 512) 255.0 1.0

mdacc_HNSCC-01-0311_CT-SIM-07-08-2007 ../data/raw-data/scans/mdacc_HNSCC-01-0311_CT-SIM-07-08-2007-_raw_raw_raw_xx.nrrd 169 ../data/train/train_selection/selection_npy/train/
69 th mdacc_HNSCC-01-0311_CT-SIM-07-08-2007 (256, 512, 512) 255.0 1.0

mdacc_HNSCC-01-0313_CT-SIM-07-08-2007 ../data/raw-data/scans/mdacc_HNSCC-01-0313_CT-SIM-07-08-2007-_raw_raw_raw_xx.nrrd 117 ../data/train/train_selection/selection_npy/train/
70 th mdacc_HNSCC-01-0313_CT-SIM-07-08-2007 (248, 512, 512) 255.0 1.0

mdacc_HNSCC-01-0314_CT-SIM-07-08-2007 ../data/raw-data/scans/mdacc_HNSCC-01-0314_CT-SIM-07-08-2007-_raw_raw_raw_xx.nrrd 213 ../data/train/train_selection/selection_npy/train/
71 th mdacc_HNSCC-01-0314_CT-SIM-07-08-2007 (338, 512, 512) 255.0 1.0

mdacc_HNSCC-01-0315_CT-SIM-07-08-2007 ../data/raw-data/scans/mdacc_HNSCC-01-0315_CT-SIM-07-08-2007-_raw_raw_raw_xx.nrrd 141 ../data/train/train_selection/selection_npy/train/
72 th mdacc_HNSC

102 th mdacc_HNSCC-01-0352_CT-SIM-08-10-2008 (287, 512, 512) 255.0 1.0

mdacc_HNSCC-01-0355_CT-SIM-08-10-2008 ../data/raw-data/scans/mdacc_HNSCC-01-0355_CT-SIM-08-10-2008-_raw_raw_raw_xx.nrrd 142 ../data/train/train_selection/selection_npy/train/
103 th mdacc_HNSCC-01-0355_CT-SIM-08-10-2008 (250, 512, 512) 255.0 1.0

mdacc_HNSCC-01-0356_CT-SIM-08-10-2008 ../data/raw-data/scans/mdacc_HNSCC-01-0356_CT-SIM-08-10-2008-_raw_raw_raw_xx.nrrd 147 ../data/train/train_selection/selection_npy/train/
104 th mdacc_HNSCC-01-0356_CT-SIM-08-10-2008 (273, 512, 512) 255.0 1.0

mdacc_HNSCC-01-0357_CT-SIM-08-10-2008 ../data/raw-data/scans/mdacc_HNSCC-01-0357_CT-SIM-08-10-2008-_raw_raw_raw_xx.nrrd 158 ../data/train/train_selection/selection_npy/train/
105 th mdacc_HNSCC-01-0357_CT-SIM-08-10-2008 (359, 512, 512) 255.0 1.0

mdacc_HNSCC-01-0358_CT-SIM-08-10-2008 ../data/raw-data/scans/mdacc_HNSCC-01-0358_CT-SIM-08-10-2008-_raw_raw_raw_xx.nrrd 155 ../data/train/train_selection/selection_npy/train/
106 th mdacc


mdacc_HNSCC-01-0396_CT-SIM-09-13-2009 ../data/raw-data/scans/mdacc_HNSCC-01-0396_CT-SIM-09-13-2009-_raw_raw_raw_xx.nrrd 149 ../data/train/train_selection/selection_npy/train/
136 th mdacc_HNSCC-01-0396_CT-SIM-09-13-2009 (276, 512, 512) 255.0 1.0

mdacc_HNSCC-01-0397_CT-SIM-09-13-2009 ../data/raw-data/scans/mdacc_HNSCC-01-0397_CT-SIM-09-13-2009-_raw_raw_raw_xx.nrrd 161 ../data/train/train_selection/selection_npy/train/
137 th mdacc_HNSCC-01-0397_CT-SIM-09-13-2009 (248, 512, 512) 255.0 1.0

mdacc_HNSCC-01-0398_CT-SIM-09-13-2009 ../data/raw-data/scans/mdacc_HNSCC-01-0398_CT-SIM-09-13-2009-_raw_raw_raw_xx.nrrd 144 ../data/train/train_selection/selection_npy/train/
138 th mdacc_HNSCC-01-0398_CT-SIM-09-13-2009 (243, 512, 512) 255.0 1.0

mdacc_HNSCC-01-0400_CT-SIM-09-13-2009 ../data/raw-data/scans/mdacc_HNSCC-01-0400_CT-SIM-09-13-2009-_raw_raw_raw_xx.nrrd 145 ../data/train/train_selection/selection_npy/train/
139 th mdacc_HNSCC-01-0400_CT-SIM-09-13-2009 (352, 512, 512) 255.0 1.0

mdacc_HNSCC

169 th mdacc_HNSCC-01-0438_CT-SIM-09-13-2009 (282, 512, 512) 255.0 1.0

mdacc_HNSCC-01-0439_CT-SIM-09-13-2009 ../data/raw-data/scans/mdacc_HNSCC-01-0439_CT-SIM-09-13-2009-_raw_raw_raw_xx.nrrd 148 ../data/train/train_selection/selection_npy/train/
170 th mdacc_HNSCC-01-0439_CT-SIM-09-13-2009 (356, 512, 512) 255.0 1.0

mdacc_HNSCC-01-0441_CT-SIM-09-13-2009 ../data/raw-data/scans/mdacc_HNSCC-01-0441_CT-SIM-09-13-2009-_raw_raw_raw_xx.nrrd 210 ../data/train/train_selection/selection_npy/train/
171 th mdacc_HNSCC-01-0441_CT-SIM-09-13-2009 (361, 512, 512) 255.0 1.0

mdacc_HNSCC-01-0442_CT-SIM-09-13-2009 ../data/raw-data/scans/mdacc_HNSCC-01-0442_CT-SIM-09-13-2009-_raw_raw_raw_xx.nrrd 145 ../data/train/train_selection/selection_npy/train/
172 th mdacc_HNSCC-01-0442_CT-SIM-09-13-2009 (243, 512, 512) 255.0 1.0

mdacc_HNSCC-01-0443_CT-SIM-08-10-2008 ../data/raw-data/scans/mdacc_HNSCC-01-0443_CT-SIM-08-10-2008-_raw_raw_raw_xx.nrrd 182 ../data/train/train_selection/selection_npy/train/
173 th mdacc


mdacc_HNSCC-01-0483_CT-SIM-10-10-2010 ../data/raw-data/scans/mdacc_HNSCC-01-0483_CT-SIM-10-10-2010-_raw_raw_raw_xx.nrrd 139 ../data/train/train_selection/selection_npy/train/
203 th mdacc_HNSCC-01-0483_CT-SIM-10-10-2010 (265, 512, 512) 255.0 1.0

mdacc_HNSCC-01-0484_CT-SIM-10-10-2010 ../data/raw-data/scans/mdacc_HNSCC-01-0484_CT-SIM-10-10-2010-_raw_raw_raw_xx.nrrd 52 ../data/train/train_selection/selection_npy/train/
204 th mdacc_HNSCC-01-0484_CT-SIM-10-10-2010 (79, 512, 512) 255.0 3.0

mdacc_HNSCC-01-0485_CT-SIM-10-10-2010 ../data/raw-data/scans/mdacc_HNSCC-01-0485_CT-SIM-10-10-2010-_raw_raw_raw_xx.nrrd 132 ../data/train/train_selection/selection_npy/train/
205 th mdacc_HNSCC-01-0485_CT-SIM-10-10-2010 (246, 512, 512) 255.0 1.0

mdacc_HNSCC-01-0486_CT-SIM-10-10-2010 ../data/raw-data/scans/mdacc_HNSCC-01-0486_CT-SIM-10-10-2010-_raw_raw_raw_xx.nrrd 166 ../data/train/train_selection/selection_npy/train/
206 th mdacc_HNSCC-01-0486_CT-SIM-10-10-2010 (296, 512, 512) 255.0 1.0

mdacc_HNSCC-0

236 th mdacc_HNSCC-01-0528_CT-SIM-10-10-2010 (231, 512, 512) 255.0 1.0

mdacc_HNSCC-01-0529_CT-SIM-10-10-2010 ../data/raw-data/scans/mdacc_HNSCC-01-0529_CT-SIM-10-10-2010-_raw_raw_raw_xx.nrrd 132 ../data/train/train_selection/selection_npy/train/
237 th mdacc_HNSCC-01-0529_CT-SIM-10-10-2010 (272, 512, 512) 255.0 1.0

mdacc_HNSCC-01-0530_CT-SIM-10-10-2010 ../data/raw-data/scans/mdacc_HNSCC-01-0530_CT-SIM-10-10-2010-_raw_raw_raw_xx.nrrd 141 ../data/train/train_selection/selection_npy/train/
238 th mdacc_HNSCC-01-0530_CT-SIM-10-10-2010 (349, 512, 512) 255.0 1.0

mdacc_HNSCC-01-0532_CT-SIM-10-10-2010 ../data/raw-data/scans/mdacc_HNSCC-01-0532_CT-SIM-10-10-2010-_raw_raw_raw_xx.nrrd 149 ../data/train/train_selection/selection_npy/train/
239 th mdacc_HNSCC-01-0532_CT-SIM-10-10-2010 (299, 512, 512) 255.0 1.0

mdacc_HNSCC-01-0534_CT-SIM-10-10-2010 ../data/raw-data/scans/mdacc_HNSCC-01-0534_CT-SIM-10-10-2010-_raw_raw_raw_xx.nrrd 128 ../data/train/train_selection/selection_npy/train/
240 th mdacc


mdacc_HNSCC-01-0572_CT-SIM-11-13-2011 ../data/raw-data/scans/mdacc_HNSCC-01-0572_CT-SIM-11-13-2011-_raw_raw_raw_xx.nrrd 151 ../data/train/train_selection/selection_npy/train/
270 th mdacc_HNSCC-01-0572_CT-SIM-11-13-2011 (275, 512, 512) 255.0 1.0

mdacc_HNSCC-01-0574_CT-SIM-11-13-2011 ../data/raw-data/scans/mdacc_HNSCC-01-0574_CT-SIM-11-13-2011-_raw_raw_raw_xx.nrrd 121 ../data/train/train_selection/selection_npy/train/
271 th mdacc_HNSCC-01-0574_CT-SIM-11-13-2011 (272, 512, 512) 255.0 1.0

mdacc_HNSCC-01-0576_CT-SIM-11-13-2011 ../data/raw-data/scans/mdacc_HNSCC-01-0576_CT-SIM-11-13-2011-_raw_raw_raw_xx.nrrd 75 ../data/train/train_selection/selection_npy/train/
272 th mdacc_HNSCC-01-0576_CT-SIM-11-13-2011 (100, 512, 512) 255.0 2.5

mdacc_HNSCC-01-0578_CT-SIM-11-13-2011 ../data/raw-data/scans/mdacc_HNSCC-01-0578_CT-SIM-11-13-2011-_raw_raw_raw_xx.nrrd 49 ../data/train/train_selection/selection_npy/train/
273 th mdacc_HNSCC-01-0578_CT-SIM-11-13-2011 (98, 512, 512) 255.0 3.0

mdacc_HNSCC-01

In [5]:
#### Slice prediction model data preparation - VALIDATION
# The z-offset for a slice should represent its offset above or below the C3 slice in mm 
# Slices below the chosen C3 slice should be given negative offsets.
df_to_be_writen = pd.DataFrame()
df_train = df_all[df_all.Datatype.isin(['val'])]
image_dir = '../data/raw-data/scans/'
targ_dir = '../data/train/train_prediction/prediction_npy/val/'
csv_write_path = '../data/train/train_prediction/prediction_meta/val.csv'
j = 0
for idx in range(df_train.shape[0]):
    image_path = image_dir + df_train.iloc[idx,0]+'.nrrd'
    patient_id =df_train.iloc[idx,0][:37]
    c3_slice = df_train.iloc[idx,1]
    print(patient_id,image_path,c3_slice,targ_dir)
     
    image_sitk =  sitk.ReadImage(image_path)
    image_array  = sitk.GetArrayFromImage(image_sitk)
    spacing =  image_sitk.GetSpacing()[2]
    
    image_array = apply_window(image_array)
    windowed_images = image_array

    print(idx+1,'th',patient_id ,image_array.shape,np.max(image_array))

    resize_func = functools.partial(resize, output_shape=[256,256],
                                    preserve_range=True, anti_aliasing=True, mode='constant')
    series = np.dstack([resize_func(im) for im in windowed_images])
    series = np.transpose(series[:, :, :, np.newaxis], [2, 0, 1, 3])
#     print(type(series),series.shape)
    for slice_idx in range(image_array.shape[0]):
        offset = spacing*(slice_idx-c3_slice)
        offset = round(offset,5)
        npy_name = str(j).zfill(6)+'_'+patient_id+'.npy'
        npy_path = targ_dir +npy_name
        im_array = series[slice_idx,:,:,:].astype(np.uint8)
        np.save(npy_path,im_array)
#         print(npy_path ,im_array.shape,'spacing',spacing)

        df_offset = pd.DataFrame({'NPY_name':npy_name,
                       'ZOffset':offset},index=[0])
        df_to_be_writen = pd.concat([df_to_be_writen,df_offset])
        df_to_be_writen.to_csv(csv_write_path)   
        j=j+1
    print()


mdacc_HNSCC-01-0216_CT-SIM-08-10-2008 ../data/raw-data/scans/mdacc_HNSCC-01-0216_CT-SIM-08-10-2008-_raw_raw_raw_xx.nrrd 192 ../data/train/train_selection/selection_npy/val/
1 th mdacc_HNSCC-01-0216_CT-SIM-08-10-2008 (287, 512, 512) 255.0

mdacc_HNSCC-01-0221_CT-SIM-08-10-2008 ../data/raw-data/scans/mdacc_HNSCC-01-0221_CT-SIM-08-10-2008-_raw_raw_raw_xx.nrrd 161 ../data/train/train_selection/selection_npy/val/
2 th mdacc_HNSCC-01-0221_CT-SIM-08-10-2008 (271, 512, 512) 255.0

mdacc_HNSCC-01-0231_CT-SIM-05-08-2005 ../data/raw-data/scans/mdacc_HNSCC-01-0231_CT-SIM-05-08-2005-_raw_raw_raw_xx.nrrd 188 ../data/train/train_selection/selection_npy/val/
3 th mdacc_HNSCC-01-0231_CT-SIM-05-08-2005 (288, 512, 512) 255.0

mdacc_HNSCC-01-0236_CT-SIM-05-08-2005 ../data/raw-data/scans/mdacc_HNSCC-01-0236_CT-SIM-05-08-2005-_raw_raw_raw_xx.nrrd 160 ../data/train/train_selection/selection_npy/val/
4 th mdacc_HNSCC-01-0236_CT-SIM-05-08-2005 (250, 512, 512) 255.0

mdacc_HNSCC-01-0241_CT-SIM-05-08-2005 ../dat

35 th mdacc_HNSCC-01-0399_CT-SIM-09-13-2009 (217, 512, 512) 255.0

mdacc_HNSCC-01-0404_CT-SIM-09-13-2009 ../data/raw-data/scans/mdacc_HNSCC-01-0404_CT-SIM-09-13-2009-_raw_raw_raw_xx.nrrd 61 ../data/train/train_selection/selection_npy/val/
36 th mdacc_HNSCC-01-0404_CT-SIM-09-13-2009 (119, 512, 512) 255.0

mdacc_HNSCC-01-0409_CT-SIM-09-13-2009 ../data/raw-data/scans/mdacc_HNSCC-01-0409_CT-SIM-09-13-2009-_raw_raw_raw_xx.nrrd 154 ../data/train/train_selection/selection_npy/val/
37 th mdacc_HNSCC-01-0409_CT-SIM-09-13-2009 (239, 512, 512) 255.0

mdacc_HNSCC-01-0414_CT-SIM-09-13-2009 ../data/raw-data/scans/mdacc_HNSCC-01-0414_CT-SIM-09-13-2009-_raw_raw_raw_xx.nrrd 174 ../data/train/train_selection/selection_npy/val/
38 th mdacc_HNSCC-01-0414_CT-SIM-09-13-2009 (285, 512, 512) 255.0

mdacc_HNSCC-01-0419_CT-SIM-09-13-2009 ../data/raw-data/scans/mdacc_HNSCC-01-0419_CT-SIM-09-13-2009-_raw_raw_raw_xx.nrrd 176 ../data/train/train_selection/selection_npy/val/
39 th mdacc_HNSCC-01-0419_CT-SIM-09-13-20


mdacc_HNSCC-01-0584_CT-SIM-11-13-2011 ../data/raw-data/scans/mdacc_HNSCC-01-0584_CT-SIM-11-13-2011-_raw_raw_raw_xx.nrrd 159 ../data/train/train_selection/selection_npy/val/
70 th mdacc_HNSCC-01-0584_CT-SIM-11-13-2011 (303, 512, 512) 255.0



In [1]:
#### Segmentation model data preparation 

import pandas as pd
from pprint import pprint
df_c3_annotation = pd.read_csv('../data/raw-data/Slice_Annotation.csv', index_col = 0)

print(df_c3_annotation.shape)
(df_c3_annotation.head())

(390, 5)


,patient_id,C3_Manual_slice,Datatype,Z_spacing,XY_spacing
0,mdacc_HNSCC-01-0216_CT-SIM-08-10-2008-_raw_raw...,192,val,NaN,NaN
0,mdacc_HNSCC-01-0217_CT-SIM-12-16-2012-_raw_raw...,131,train,NaN,NaN
0,mdacc_HNSCC-01-0218_CT-SIM-09-13-2009-_raw_raw...,166,train,NaN,NaN
0,mdacc_HNSCC-01-0219_CT-SIM-09-13-2009-_raw_raw...,150,train,NaN,NaN
0,mdacc_HNSCC-01-0220_CT-SIM-11-13-2011-_raw_raw...,129,train,NaN,NaN


In [2]:
import os
import glob
import random

#image_folder ='../ct_scan_in_nifti/'
#mask_folder = '../ct_scan_C3_expert_segmentation_in_nifti/'
image_folder ='../data/raw-data/scans-normalized/'
mask_folder = '../data/raw-data/segs-normalized/'

images = sorted(glob.glob(os.path.join(image_folder)+'/*.nrrd'))
masks = sorted(glob.glob(os.path.join(mask_folder)+'/*.nrrd'))
[train_images, train_masks, val_images, val_masks] = [[],[],[],[]]
print('The number of CT images:',len(images),'The number of segmentation masks:',len(masks))

train_slices = []
val_slices = []

for idx,row in df_c3_annotation.iterrows():
    image_name = row['patient_id']+'.nrrd'
    mask_name = row['patient_id']+'.seg.nrrd'
    if os.path.exists(image_folder+image_name) and os.path.exists(mask_folder+mask_name):
        if (row['Datatype'] == 'train'):
            train_images.append(image_folder+image_name)
            train_masks.append(mask_folder+mask_name)
            train_slices.append(row['C3_Manual_slice'])
        if (row['Datatype'] == 'val'):
            val_images.append(image_folder+image_name)
            val_masks.append(mask_folder+mask_name)
            val_slices.append(row['C3_Manual_slice'])

print(len(train_images),len(train_masks),len(train_slices))
print(len(val_images),len(val_masks),len(val_slices))

The number of CT images: 390 The number of segmentation masks: 390
247 247 247
62 62 62


In [3]:
import numpy as np
import SimpleITK as sitk

train_images_array =  np.zeros((len(train_images),512,512,1))
val_images_array  = np.zeros((len(val_images),512,512,1))

train_masks_array  = np.zeros((len(train_images),512,512,1))
val_masks_array  = np.zeros((len(val_images),512,512,1))        

In [4]:
images = train_images
labels = train_masks

for i in range(len(images)):
    if (images[i].split('/')[-1][:38]==labels[i].split('/')[-1][:38]):
        print(i+1,'th processing:___'+images[i].split('/')[-1][:38])
        image_sitk = sitk.ReadImage(images[i])
        label_sitk = sitk.ReadImage(labels[i])
        
        seg_slice = train_slices[i]
        
        image_array  = sitk.GetArrayFromImage(image_sitk)[seg_slice,:,:].reshape(1,512,512,1)
        label_array  = sitk.GetArrayFromImage(label_sitk )[seg_slice,:,:].reshape(1,512,512,1)
        print(np.sum(label_array))
        train_images_array[i,:,:,:] = image_array
        train_masks_array[i,:,:,:] = label_array
        print()
train_masks_array=train_masks_array.astype(np.uint8)

np.save('../data/train/train_segmentation/train_images.npy',train_images_array)
np.save('../data/train/train_segmentation/train_masks.npy',train_masks_array)

1 th processing:___mdacc_HNSCC-01-0217_CT-SIM-12-16-2012-
12591.0

2 th processing:___mdacc_HNSCC-01-0218_CT-SIM-09-13-2009-
20226.0

3 th processing:___mdacc_HNSCC-01-0219_CT-SIM-09-13-2009-
12748.0

4 th processing:___mdacc_HNSCC-01-0220_CT-SIM-11-13-2011-
15063.0

5 th processing:___mdacc_HNSCC-01-0222_CT-SIM-09-13-2009-
10958.0

6 th processing:___mdacc_HNSCC-01-0223_CT-SIM-10-10-2010-
11981.0

7 th processing:___mdacc_HNSCC-01-0224_CT-SIM-07-08-2007-
12384.0

8 th processing:___mdacc_HNSCC-01-0230_CT-SIM-05-08-2005-
13431.0

9 th processing:___mdacc_HNSCC-01-0232_CT-SIM-05-08-2005-
21591.0

10 th processing:___mdacc_HNSCC-01-0233_CT-SIM-05-08-2005-
17375.0

11 th processing:___mdacc_HNSCC-01-0234_CT-SIM-05-08-2005-
19633.0

12 th processing:___mdacc_HNSCC-01-0235_CT-SIM-05-08-2005-
17977.0

13 th processing:___mdacc_HNSCC-01-0237_CT-SIM-05-08-2005-
16415.0

14 th processing:___mdacc_HNSCC-01-0238_CT-SIM-05-08-2005-
21586.0

15 th processing:___mdacc_HNSCC-01-0239_CT-SIM-05-08-2005

21151.0

122 th processing:___mdacc_HNSCC-01-0378_CT-SIM-08-10-2008-
15499.0

123 th processing:___mdacc_HNSCC-01-0380_CT-SIM-08-10-2008-
15997.0

124 th processing:___mdacc_HNSCC-01-0381_CT-SIM-08-10-2008-
23179.0

125 th processing:___mdacc_HNSCC-01-0382_CT-SIM-08-10-2008-
15717.0

126 th processing:___mdacc_HNSCC-01-0383_CT-SIM-08-10-2008-
18634.0

127 th processing:___mdacc_HNSCC-01-0385_CT-SIM-08-10-2008-
20751.0

128 th processing:___mdacc_HNSCC-01-0386_CT-SIM-08-10-2008-
13405.0

129 th processing:___mdacc_HNSCC-01-0387_CT-SIM-08-10-2008-
20947.0

130 th processing:___mdacc_HNSCC-01-0388_CT-SIM-08-10-2008-
17720.0

131 th processing:___mdacc_HNSCC-01-0390_CT-SIM-08-10-2008-
21074.0

132 th processing:___mdacc_HNSCC-01-0391_CT-SIM-08-10-2008-
18366.0

133 th processing:___mdacc_HNSCC-01-0392_CT-SIM-08-10-2008-
22354.0

134 th processing:___mdacc_HNSCC-01-0393_CT-SIM-08-10-2008-
18446.0

135 th processing:___mdacc_HNSCC-01-0395_CT-SIM-08-10-2008-
22223.0

136 th processing:___mdac

20736.0

241 th processing:___mdacc_HNSCC-01-0535_CT-SIM-10-10-2010-
13584.0

242 th processing:___mdacc_HNSCC-01-0536_CT-SIM-10-10-2010-
12385.0

243 th processing:___mdacc_HNSCC-01-0538_CT-SIM-10-10-2010-
19519.0

244 th processing:___mdacc_HNSCC-01-0539_CT-SIM-10-10-2010-
17992.0

245 th processing:___mdacc_HNSCC-01-0540_CT-SIM-10-10-2010-
11791.0

246 th processing:___mdacc_HNSCC-01-0541_CT-SIM-10-10-2010-
21882.0

247 th processing:___mdacc_HNSCC-01-0543_CT-SIM-11-13-2011-
19743.0



In [5]:
images = val_images
labels = val_masks
for i in range(len(images)):
    if (images[i].split('/')[-1][:20]==labels[i].split('/')[-1][:20] ):
        print(i+1,'th processing:   '+images[i].split('/')[-1][:20])
        image_sitk = sitk.ReadImage(images[i])
        label_sitk = sitk.ReadImage(labels[i])
        
        seg_slice = val_slices[i]
        image_array  = sitk.GetArrayFromImage(image_sitk)[seg_slice,:,:].reshape(1,512,512,1)
        label_array  = sitk.GetArrayFromImage(label_sitk )[seg_slice,:,:].reshape(1,512,512,1)
        print(np.sum(label_array))
        val_images_array[i,:,:,:] = image_array
        val_masks_array[i,:,:,:] = label_array
        print()
 
    val_masks_array=val_masks_array.astype(np.uint8)
np.save('../data/train/train_segmentation/val_images.npy',val_images_array)
np.save('../data/train/train_segmentation/val_masks.npy',val_masks_array)

1 th processing:   mdacc_HNSCC-01-0216_
19560.0

2 th processing:   mdacc_HNSCC-01-0221_
16956.0

3 th processing:   mdacc_HNSCC-01-0231_
21771.0

4 th processing:   mdacc_HNSCC-01-0236_
19808.0

5 th processing:   mdacc_HNSCC-01-0241_
23254.0

6 th processing:   mdacc_HNSCC-01-0246_
20549.0

7 th processing:   mdacc_HNSCC-01-0251_
36684.0

8 th processing:   mdacc_HNSCC-01-0256_
18848.0

9 th processing:   mdacc_HNSCC-01-0261_
21176.0

10 th processing:   mdacc_HNSCC-01-0266_
14545.0

11 th processing:   mdacc_HNSCC-01-0272_
18370.0

12 th processing:   mdacc_HNSCC-01-0277_
20085.0

13 th processing:   mdacc_HNSCC-01-0284_
16501.0

14 th processing:   mdacc_HNSCC-01-0289_
18646.0

15 th processing:   mdacc_HNSCC-01-0294_
14972.0

16 th processing:   mdacc_HNSCC-01-0301_
16007.0

17 th processing:   mdacc_HNSCC-01-0306_
20635.0

18 th processing:   mdacc_HNSCC-01-0312_
18430.0

19 th processing:   mdacc_HNSCC-01-0317_
20419.0

20 th processing:   mdacc_HNSCC-01-0322_
18921.0

21 th pro